In [2]:
import google.generativeai as genai
import anthropic
from anthropic import Anthropic
from google.generativeai import GenerativeModel
gemini_api_key = "AI4"
claude_api_key = "sA"

In [3]:
gemini_system_inst = """You are an expert on the specific data provided to you. 
Your task is to answer queries related to that data with precision and accuracy. 
You must base your answers strictly on the information within the knowledge base provided, using only facts, correlations,
or deductions that can be clearly linked to the content. Do not provide information or reasoning beyond the scope of this data.
If a question cannot be answered based on the available data, 
respond by stating that the necessary information is not available."""

In [4]:
claude_system_inst = """
    You are a conversational model, designed to provide smart and precise answers. 
    However, you do not have knowledge of everything that a user may ask.
    When you encounter a question where you lack sufficient information or are unsure,
    you must output only the words "MEMORY BUS"—nothing more, nothing less.

    Once you output "MEMORY BUS," you will be prompted with the question, "What is your question?" At that point,
    you should ask a question related to the missing information you need, specifically tied to the user's original query. 
    For example, if the user asks about a research paper titled 'ABC 2021' and you do not know about it, you will type "MEMORY BUS" 
    and then ask, "What is the paper ABC 2021?"

    After receiving the answer from the knowledge base, you will be provided with the relevant data and the
    original user's question again. Using the retrieved information, along with the user’s original question, 
    you will then proceed to answer the user's query properly and accurately.

    You may use the newly retrieved context for subsequent conversations. However, if you need to access the knowledge base again 
    for further clarification or additional information, repeat the process by typing "MEMORY BUS" and following the same steps.
    """

In [5]:
genai.configure(api_key = gemini_api_key )
client = anthropic.Anthropic(
    api_key=claude_api_key,
)

In [6]:
generation_config = {
        "temperature": 0.3,
        "top_p": 0.95,
        "top_k": 64,
        "max_output_tokens": 8192,
    }
model = genai.GenerativeModel(
        model_name="gemini-1.5-flash",
        system_instruction= gemini_system_inst,
        generation_config=generation_config,
    )

In [19]:
def get_ai_response(client: Anthropic, messages, model: str, system: str) -> str:
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        system=system
    )
    return response.content[0].text

def chat_loop(client: Anthropic, model: GenerativeModel, claude_system_inst: str, sample_pdf: str):
    messages = []
    
    while True:
        user_input = input("talk (or '1' to exit): ")
        if user_input == "1":
            break
        
        messages.append({"role": "user", "content": user_input})
        
        response = get_ai_response(client, messages, "claude-3-haiku-20240307", claude_system_inst)
        messages.append({"role": "assistant", "content": response})
        
        if "memory bus" in response.lower():
            #print("claude said->", response)
            print("Accessing knowledge base...")
            
            messages.append({"role": "user", "content": """What do you want to know from the knowledge base?
            (you will be talking to a llm, not the user here so word yourself precise adhering to that)"""})
            
            query = get_ai_response(client, messages, "claude-3-haiku-20240307", claude_system_inst)
            #print("query is -> ", query)
            messages.append({"role": "assistant", "content": query})
            
            pdf_info = model.generate_content([f"Answer this from the pdf file. {query}", sample_pdf]).text
            enhanced_input = f"{pdf_info}\n{user_input}"
            #print("gemini said-> ",enhanced_input)
            
            messages.append({"role": "user", "content": enhanced_input})
            final_response = get_ai_response(client, messages, "claude-3-haiku-20240307", claude_system_inst)
            messages.append({"role": "assistant", "content": final_response})
            
            print("\nAssistant-> ", final_response,"\n")
        else:
            print("\nAssistant-> ",response, "\n")

In [14]:
#uploading knowledge base
sample_pdf = genai.upload_file("test2.pdf")

In [20]:
chat_loop(client, model, claude_system_inst, sample_pdf)

talk (or '1' to exit):  hello



Assistant->  Hello! How can I assist you today? 



talk (or '1' to exit):  tell me about the rational design of large anomalous Nernst effect in Dirac semimetals.


Accessing knowledge base...

Assistant->  Thank you for providing the requested information from the knowledge base. With this context, I can now provide a more detailed response to the original query about the rational design of large anomalous Nernst effect in Dirac semimetals.

The key characteristics of Dirac semimetals that enable the anomalous Nernst effect are their unique band structures featuring multiple Weyl points or nodal lines. These band crossings in momentum space correspond to divergences in the Berry curvature, which is a critical component for generating the anomalous Nernst effect.

The proposed design principle for achieving a large anomalous Nernst effect in Dirac semimetals involves creating a pair of Dirac nodes under a Zeeman field. This configuration leads to an odd-distributed, double-peak anomalous Hall conductivity curve with respect to the chemical potential. This specific band structure results in an enhanced anomalous Nernst conductivity, which is the ke

talk (or '1' to exit):  1


In [21]:
#deleting knowledge base after use
sample_pdf.delete()